# Pull raw Kobo JSON record 136

This notebook connects to KoboToolbox, finds the form `gaia_metadata_authoring_form_v2`, pulls all submissions as raw JSON, and saves the raw record where `control_group/id = 136`.

No pandas DataFrame conversion is used.


In [ ]:
"""
Pull raw KoboToolbox JSON and extract record ID 136.

This script:
1. Connects to KoboToolbox using an API token.
2. Finds the Kobo form named gaia_metadata_authoring_form_v2.
3. Pulls all submissions from the form.
4. Saves the full raw Kobo response as JSON.
5. Extracts the raw JSON record where control_group/id == 136.
6. Saves the extracted record as record_136_raw.json.
7. Saves the record keys to help with JSONPath extraction.

No pandas DataFrame conversion is used.
"""

import os
import json
import requests
from getpass import getpass
from pathlib import Path


In [ ]:
# 1. Configuration


In [ ]:
SERVER_URL = "https://kf.kobotoolbox.org"
TARGET_FORM_NAME = "gaia_metadata_authoring_form_v2"
TARGET_ID = "136"

# Output folder will be created in the folder where you run this script/notebook
OUTPUT_DIR = Path.cwd() / "kobo_output"

RAW_RESPONSE_FILE = OUTPUT_DIR / "raw_kobo_response.json"
TARGET_RECORD_FILE = OUTPUT_DIR / f"record_{TARGET_ID}_raw.json"
TARGET_RECORD_KEYS_FILE = OUTPUT_DIR / f"record_{TARGET_ID}_keys.txt"


In [ ]:
# 2. Authentication


In [ ]:
# Option A: use environment variable KOBO_API_TOKEN
# Option B: paste token when prompted
API_TOKEN = (os.getenv("KOBO_API_TOKEN") or getpass("Paste Kobo API token: ")).strip()

headers = {
    "Authorization": f"Token {API_TOKEN}",
    "Accept": "application/json",
}


In [ ]:
# 3. Helper functions


In [ ]:
def get_json(url, headers, timeout=60):
    """GET a URL and return parsed JSON."""
    response = requests.get(url, headers=headers, timeout=timeout)
    print(f"GET {response.url} -> {response.status_code}")
    response.raise_for_status()
    return response.json()


def fetch_paginated_results(start_url, headers):
    """Fetch all paginated results from a Kobo v2 API endpoint."""
    results = []
    next_url = start_url

    while next_url:
        payload = get_json(next_url, headers=headers)
        results.extend(payload.get("results", []))
        next_url = payload.get("next")

    return results


def get_record_id(record):
    """Return the legacy/control ID from a Kobo record.

    Kobo can store grouped question names as slash-separated keys such as:
        control_group/id

    This function also checks a few fallback patterns in case the export
    structure changes.
    """
    candidate_keys = [
        "control_group/id",
        "id",
        "ID",
        "dataset_id",
        "_id",
    ]

    for key in candidate_keys:
        if key in record and record.get(key) not in [None, ""]:
            return str(record.get(key))

    # Fallback for nested group structure, if Kobo returns it that way
    control_group = record.get("control_group")
    if isinstance(control_group, dict) and control_group.get("id") not in [None, ""]:
        return str(control_group.get("id"))

    return None


In [ ]:
# 4. Show where output will be saved


In [ ]:
print("Current working directory:")
print(Path.cwd())

print("\nOutput folder:")
print(OUTPUT_DIR)


In [ ]:
# 5. Fetch all Kobo forms/assets


In [ ]:
assets_url = f"{SERVER_URL}/api/v2/assets/"
assets = fetch_paginated_results(assets_url, headers=headers)

print(f"\nTotal assets found: {len(assets)}")


In [ ]:
# 6. Find the target Kobo form


In [ ]:
target_asset = None
target_form_name_normalized = TARGET_FORM_NAME.strip().lower()

for asset in assets:
    asset_name = asset.get("name", "").strip().lower()

    if asset_name == target_form_name_normalized:
        target_asset = asset
        print(f"\nSuccess! Found form: {asset.get('name')}")
        print(f"UID: {asset.get('uid')}")
        print(f"Data endpoint: {asset.get('data')}")
        break

if target_asset is None:
    available_names = sorted([a.get("name", "") for a in assets if a.get("name")])

    print("\nAvailable form names:")
    for name in available_names:
        print("-", name)

    raise ValueError(f"Could not find a form named '{TARGET_FORM_NAME}'")


In [ ]:
# 7. Pull all submissions from the target form


In [ ]:
data_url = target_asset.get("data")

if not data_url:
    raise ValueError("The target Kobo form does not have a data endpoint.")

submissions = fetch_paginated_results(data_url, headers=headers)

print(f"\nTotal submissions retrieved: {len(submissions)}")


In [ ]:
# 8. Save the full raw Kobo response


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_payload = {
    "form_name": target_asset.get("name"),
    "form_uid": target_asset.get("uid"),
    "count": len(submissions),
    "results": submissions,
}

with open(RAW_RESPONSE_FILE, "w", encoding="utf-8") as f:
    json.dump(raw_payload, f, indent=2, ensure_ascii=False)

print("\nSaved full raw Kobo response to:")
print(RAW_RESPONSE_FILE.resolve())


In [ ]:
# 9. Extract the raw JSON record where control_group/id == TARGET_ID


In [ ]:
target_record = None

for record in submissions:
    if get_record_id(record) == TARGET_ID:
        target_record = record
        break

if target_record is None:
    found_ids = [get_record_id(record) for record in submissions]
    found_ids = [x for x in found_ids if x is not None]

    print("\nIDs found:")
    print(found_ids[:50])

    raise ValueError(f"No raw JSON record found for ID {TARGET_ID}")

with open(TARGET_RECORD_FILE, "w", encoding="utf-8") as f:
    json.dump(target_record, f, indent=2, ensure_ascii=False)

print(f"\nSaved raw JSON record for ID {TARGET_ID} to:")
print(TARGET_RECORD_FILE.resolve())


In [ ]:
# 10. Save and print the keys in the extracted record


In [ ]:
record_keys = sorted(target_record.keys())

with open(TARGET_RECORD_KEYS_FILE, "w", encoding="utf-8") as f:
    for key in record_keys:
        f.write(key + "\n")

print("\nSaved record key list to:")
print(TARGET_RECORD_KEYS_FILE.resolve())

print("\nFirst 50 keys in record 136:")
for key in record_keys[:50]:
    print("-", key)


In [ ]:
# 11. Quick check of record 136


In [ ]:
print("\nQuick check:")
print("Record ID:", get_record_id(target_record))
print("Title:", target_record.get("dublin_core_group/title"))
print("DOI:", target_record.get("dublin_core_group/doi"))

description = str(target_record.get("dublin_core_group/description", ""))
print("Description preview:", description[:300])


In [ ]:
# 12. Confirm files created


In [ ]:
print("\nFiles created in kobo_output:")
for file in sorted(OUTPUT_DIR.iterdir()):
    print("-", file.name)


print("\nNext step:")
print(f"Use {TARGET_RECORD_FILE.resolve()} for JSONPath extraction.")
